<a href="https://colab.research.google.com/github/deniwidi/data-science-2026/blob/main/Pertemuan3_Deni_Widi_Alfian_240401010340.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
# ============================================
# PIPELINE DATA CLEANING - HOUSING DATASET
# ============================================

import pandas as pd
import numpy as np
from scipy.stats.mstats import winsorize
import missingno as msno
import requests
from pandas import json_normalize
import matplotlib.pyplot as plt

In [16]:
# STEP 0 - Load & eksplorasi awal

url = "https://drive.google.com/uc?export=download&id=1LfQWProB0VjWN5q8bKuRIgn-stULfIRo"
df = pd.read_csv(url)

print("Dataset berhasil dimuat!")
print("Shape awal:", df.shape)
print(df.isnull().sum())

Dataset berhasil dimuat!
Shape awal: (130, 7)
id               0
luas_m2         18
harga_juta      17
kota             0
kamar           10
tahun_bangun     0
kondisi          0
dtype: int64


In [17]:
# STEP 1 - Hapus Duplikat

n_dup = df.duplicated().sum()
print(f"Jumlah baris duplikat ditemukan: {n_dup}")

df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)

print(f"Duplikat dihapus.")
print(f"Shape setelah hapus duplikat: {df.shape}")

Jumlah baris duplikat ditemukan: 0
Duplikat dihapus.
Shape setelah hapus duplikat: (130, 7)


In [18]:
# STEP 2A - Normalisasi String & Koreksi Nama Kota

df['kota'] = df['kota'].str.strip().str.lower()

kota_mapping = {
    'jakarta'        : 'Jakarta',
    'jkt'            : 'Jakarta',
    'jak'            : 'Jakarta',
    'jakarta selatan': 'Jakarta Selatan',
    'jakarta barat'  : 'Jakarta Barat',
    'jakarta timur'  : 'Jakarta Timur',
    'jakarta utara'  : 'Jakarta Utara',
    'jakarta pusat'  : 'Jakarta Pusat',
    'depok'          : 'Depok',
    'dpk'            : 'Depok',
    'dep'            : 'Depok',
    'bekasi'         : 'Bekasi',
    'bks'            : 'Bekasi',
    'bksi'           : 'Bekasi',
    'bogor'          : 'Bogor',
    'bgr'            : 'Bogor',
    'bog'            : 'Bogor',
    'tangerang'      : 'Tangerang',
    'tgr'            : 'Tangerang',
    'tng'            : 'Tangerang',
    'tangsel'        : 'Tangerang Selatan',
    'tangerang selatan': 'Tangerang Selatan',
    'bandung'        : 'Bandung',
    'bdg'            : 'Bandung',
    'bndg'           : 'Bandung',
    'surabaya'       : 'Surabaya',
    'sby'            : 'Surabaya',
    'sbya'           : 'Surabaya',
    'makassar'       : 'Makassar',
    'makasar'        : 'Makassar',
    'mksr'           : 'Makassar',
    'mks'            : 'Makassar',
    'ujung pandang'  : 'Makassar',
    'medan'          : 'Medan',
    'mdn'            : 'Medan',
    'med'            : 'Medan',
    'semarang'       : 'Semarang',
    'smrg'           : 'Semarang',
    'smg'            : 'Semarang',
    'yogyakarta'     : 'Yogyakarta',
    'yogya'          : 'Yogyakarta',
    'jogja'          : 'Yogyakarta',
    'jgj'            : 'Yogyakarta',
    'yk'             : 'Yogyakarta',
    'ygy'            : 'Yogyakarta',
    'yog'            : 'Yogyakarta',
    'palembang'      : 'Palembang',
    'plg'            : 'Palembang',
    'plmbg'          : 'Palembang',
    'malang'         : 'Malang',
    'mlg'            : 'Malang',
    'denpasar'       : 'Denpasar',
    'bali'           : 'Denpasar',
    'dnp'            : 'Denpasar',
}

df['kota'] = df['kota'].map(kota_mapping).fillna(df['kota'].str.title())

print("Normalisasi nama kota selesai.")

Normalisasi nama kota selesai.


In [26]:
# STEP 2B - Normalisasi String & Koreksi Kategori Kondisi

# Bersihkan dan lowercase dulu
df['kondisi'] = df['kondisi'].str.strip().str.lower()

print("Nilai kondisi sebelum mapping:")
print(df['kondisi'].value_counts())

# Mapping semua variasi jadi 3 kategori: baik, sedang, buruk
kondisi_mapping = {
    # BAIK
    'baik'      : 'baik',
    'bagus'     : 'baik',
    'sangat baik': 'baik',
    'sempurna'  : 'baik',
    'baru'      : 'baik',
    'prima'     : 'baik',
    'mewah'     : 'baik',
    'terawat'   : 'baik',
    'ok'        : 'baik',
    'good'      : 'baik',
    'great'     : 'baik',
    'excellent' : 'baik',

    # SEDANG
    'sedang'    : 'sedang',
    'cukup'     : 'sedang',
    'lumayan'   : 'sedang',
    'biasa'     : 'sedang',
    'average'   : 'sedang',
    'standar'   : 'sedang',
    'normal'    : 'sedang',
    'fair'      : 'sedang',

    # BURUK
    'buruk'     : 'buruk',
    'jelek'     : 'buruk',
    'rusak'     : 'buruk',
    'parah'     : 'buruk',
    'hancur'    : 'buruk',
    'kumuh'     : 'buruk',
    'kotor'     : 'buruk',
    'bad'       : 'buruk',
    'poor'      : 'buruk',
    'tidak baik': 'buruk',
}

df['kondisi'] = df['kondisi'].map(kondisi_mapping)

# Jika ada nilai yang tidak ter-mapping, isi dengan modus
if df['kondisi'].isnull().sum() > 0:
    modus_kondisi = df['kondisi'].mode()[0]
    df['kondisi'].fillna(modus_kondisi, inplace=True)
    print(f"Nilai kondisi tidak dikenal diisi dengan modus: '{modus_kondisi}'")

print("\nNormalisasi kondisi selesai.")
print("Distribusi kondisi setelah mapping:")
print(df['kondisi'].value_counts())

Nilai kondisi sebelum mapping:
kondisi
baik      73
sedang    43
buruk     14
Name: count, dtype: int64

Normalisasi kondisi selesai.
Distribusi kondisi setelah mapping:
kondisi
baik      73
sedang    43
buruk     14
Name: count, dtype: int64


In [20]:
# STEP 3 - Imputasi Missing Values

print("Missing values sebelum imputasi:")
print(df.isnull().sum())

# Imputasi median untuk kolom numerik

df['luas_m2']    = df['luas_m2'].fillna(df['luas_m2'].median())
df['harga_juta'] = df['harga_juta'].fillna(df['harga_juta'].median())

# Imputasi modus untuk kolom kategorik/integer

df['kamar'] = df['kamar'].fillna(df['kamar'].mode()[0])

# Imputasi modus untuk kota & kondisi jika masih ada missing

if df['kota'].isnull().sum() > 0:
    df['kota'] = df['kota'].fillna(df['kota'].mode()[0])
if df['kondisi'].isnull().sum() > 0:
    df['kondisi'] = df['kondisi'].fillna(df['kondisi'].mode()[0])

print("\nMissing values setelah imputasi:")
print(df.isnull().sum())
print("\nImputasi missing values selesai.")

Missing values sebelum imputasi:
id               0
luas_m2         18
harga_juta      17
kota             0
kamar           10
tahun_bangun     0
kondisi          0
dtype: int64

Missing values setelah imputasi:
id              0
luas_m2         0
harga_juta      0
kota            0
kamar           0
tahun_bangun    0
kondisi         0
dtype: int64

Imputasi missing values selesai.


In [21]:
# STEP 4 - Tangani Outlier (IQR Fence)

def deteksi_outlier_iqr(df, kolom):
    Q1 = df[kolom].quantile(0.25)
    Q3 = df[kolom].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[kolom] < lower) | (df[kolom] > upper)]
    return lower, upper, outliers

# Fix luas_m2
LUAS_MIN = 10.0   # luas minimum yang masuk akal (m²)

print("=== luas_m2 ===")
print(f"Sebelum fix — min: {df['luas_m2'].min()}, max: {df['luas_m2'].max()}")
print(f"Nilai <= 0  : {(df['luas_m2'] <= 0).sum()} baris")

# Hitung IQR
Q1, Q3 = df['luas_m2'].quantile([0.25, 0.75])
IQR = Q3 - Q1
lower_iqr = Q1 - 1.5 * IQR
upper_iqr = Q3 + 1.5 * IQR

# Gunakan batas bawah terbesar antara IQR lower dan LUAS_MIN
lower_final = max(lower_iqr, LUAS_MIN)
upper_final = upper_iqr

print(f"Batas IQR   : [{lower_iqr:.2f}, {upper_iqr:.2f}]")
print(f"Batas final : [{lower_final:.2f}, {upper_final:.2f}]")

# Ganti nilai tidak wajar dengan median luas yang valid
median_luas = df.loc[df['luas_m2'] >= LUAS_MIN, 'luas_m2'].median()
n_invalid = (df['luas_m2'] <= 0).sum()
df.loc[df['luas_m2'] <= 0, 'luas_m2'] = median_luas
print(f"{n_invalid} nilai <= 0 diganti median ({median_luas:.1f} m²)")

# Clip sisa outlier dengan batas IQR
df['luas_m2'] = df['luas_m2'].clip(lower=lower_final, upper=upper_final)
print(f"Setelah fix — min: {df['luas_m2'].min():.2f}, max: {df['luas_m2'].max():.2f}")

# Fix harga_juta
HARGA_MIN = 50.0   # harga minimum yang masuk akal (juta)

print("\n=== harga_juta ===")
print(f"Sebelum fix — min: {df['harga_juta'].min()}, max: {df['harga_juta'].max()}")

Q1, Q3 = df['harga_juta'].quantile([0.25, 0.75])
IQR = Q3 - Q1
lower_iqr = Q1 - 1.5 * IQR
upper_iqr = Q3 + 1.5 * IQR

lower_final = max(lower_iqr, HARGA_MIN)
upper_final = upper_iqr

print(f"Batas IQR   : [{lower_iqr:.2f}, {upper_iqr:.2f}]")
print(f"Batas final : [{lower_final:.2f}, {upper_final:.2f}]")

median_harga = df.loc[df['harga_juta'] >= HARGA_MIN, 'harga_juta'].median()
n_invalid = (df['harga_juta'] <= 0).sum()
df.loc[df['harga_juta'] <= 0, 'harga_juta'] = median_harga
print(f"{n_invalid} nilai <= 0 diganti median ({median_harga:.1f} juta)")

df['harga_juta'] = df['harga_juta'].clip(lower=lower_final, upper=upper_final)
print(f"Setelah fix — min: {df['harga_juta'].min():.2f}, max: {df['harga_juta'].max():.2f}")

# Fix tahun_bangun
TAHUN_MIN = 1950
TAHUN_MAX = 2024

print("\n=== tahun_bangun ===")
print(f"Sebelum fix — min: {df['tahun_bangun'].min()}, max: {df['tahun_bangun'].max()}")

df['tahun_bangun'] = df['tahun_bangun'].round().astype('Int64')

median_tahun = int(df.loc[
    df['tahun_bangun'].between(TAHUN_MIN, TAHUN_MAX), 'tahun_bangun'
].median())

n_fix = (~df['tahun_bangun'].between(TAHUN_MIN, TAHUN_MAX)).sum()
df.loc[~df['tahun_bangun'].between(TAHUN_MIN, TAHUN_MAX), 'tahun_bangun'] = median_tahun
print(f"{n_fix} baris tahun tidak wajar → diganti median ({median_tahun})")
print(f"Setelah fix — min: {df['tahun_bangun'].min()}, max: {df['tahun_bangun'].max()}")

print("\nSemua outlier selesai ditangani.")

=== luas_m2 ===
Sebelum fix — min: -50.0, max: 9500.0
Nilai <= 0  : 1 baris
Batas IQR   : [-145.22, 512.97]
Batas final : [10.00, 512.97]
1 nilai <= 0 diganti median (193.8 m²)
Setelah fix — min: 40.40, max: 512.97

=== harga_juta ===
Sebelum fix — min: -500.0, max: 99999999.0
Batas IQR   : [-422.75, 1719.25]
Batas final : [50.00, 1719.25]
2 nilai <= 0 diganti median (655.0 juta)
Setelah fix — min: 139.00, max: 1719.25

=== tahun_bangun ===
Sebelum fix — min: 1890, max: 9999
3 baris tahun tidak wajar → diganti median (2002)
Setelah fix — min: 1980, max: 2023

Semua outlier selesai ditangani.


In [32]:
# STEP 5A - Validasi & Ekspor

total_missing = df.isnull().sum().sum()
total_duplikat = df.duplicated().sum()

assert total_missing == 0,  "Masih ada missing values!"
assert total_duplikat == 0, "Masih ada duplikat!"

In [31]:
# STEP 5A - Validasi & Ekspor

df.to_csv('housing_clean.csv', index=False)
print("Dataset bersih tersimpan sebagai 'housing_clean.csv'")
print(f"Shape final: {df.shape}")

Dataset bersih tersimpan sebagai 'housing_clean.csv'
Shape final: (130, 7)


In [24]:
# STEP 6 - Akses REST API JSONPlaceholder

import requests
from pandas import json_normalize

# ① Ambil data users
URL_USERS = "https://jsonplaceholder.typicode.com/users"

try:
    response = requests.get(URL_USERS, timeout=10)

    if response.status_code == 200:
        data_users = response.json()
        df_users = json_normalize(data_users, sep='_')
        print("Data Users berhasil diambil dari API!")
        print(f"Shape: {df_users.shape}")
        print(df_users[['id', 'name', 'email', 'address_city']].to_string())
    else:
        print(f"Error: Status code {response.status_code}")

except requests.exceptions.ConnectionError as e:
    print(f"Koneksi gagal: {e}")

# ② Ambil data posts milik user 1
URL_POSTS = "https://jsonplaceholder.typicode.com/posts"
params = {'userId': 1}

try:
    response_posts = requests.get(URL_POSTS, params=params, timeout=10)

    if response_posts.status_code == 200:
        data_posts = response_posts.json()
        df_posts = pd.DataFrame(data_posts)
        print(f"\nData Posts (userId=1) berhasil diambil!")
        print(f"Shape: {df_posts.shape}")
        print(df_posts[['userId', 'id', 'title']].head())
    else:
        print(f"Error posts: {response_posts.status_code}")

except requests.exceptions.ConnectionError as e:
    print(f"Koneksi gagal: {e}")

Data Users berhasil diambil dari API!
Shape: (10, 15)
   id                      name                      email    address_city
0   1             Leanne Graham          Sincere@april.biz     Gwenborough
1   2              Ervin Howell          Shanna@melissa.tv     Wisokyburgh
2   3          Clementine Bauch         Nathan@yesenia.net   McKenziehaven
3   4          Patricia Lebsack  Julianne.OConner@kory.org     South Elvis
4   5          Chelsey Dietrich   Lucio_Hettinger@annie.ca      Roscoeview
5   6      Mrs. Dennis Schulist    Karley_Dach@jasper.info   South Christy
6   7           Kurtis Weissnat     Telly.Hoeger@billy.biz       Howemouth
7   8  Nicholas Runolfsdottir V       Sherwood@rosamond.me       Aliyaview
8   9           Glenna Reichert    Chaim_McDermott@dana.io  Bartholomebury
9  10        Clementina DuBuque     Rey.Padberg@karina.biz     Lebsackbury

Data Posts (userId=1) berhasil diambil!
Shape: (10, 4)
   userId  id                                              title


In [33]:
from google.colab import files

files.download('housing_clean.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>